# DEF - Schleswig-Holstein

In [23]:
import os

import patoolib
import pandas as pd
from tqdm import tqdm
from camelsp import get_metadata

from camels_de1h import get_input_path, get_nuts_id_from_provider_id, Station1h

## Extract data

In [2]:
input_path = get_input_path("DEF")

# extract zip
if not (input_path / "Q_und_W").exists():
    os.makedirs(input_path / "Q_und_W")
    patoolib.extract_archive(str(input_path / "20250304_Pegeldaten.7z"), outdir=str(input_path / "Q_und_W"), interactive=False, verbosity=-1)
    print("Data extracted.")
else:
    print("Data already extracted.")

Data already extracted.


## Parse data

**Timezone: UTC+1 (MEZ)**

Most of the data is in hourly resolution, we received data for two gauges in 15 minute and in hourly resolution.  
As these are only two gauges, we ignore the 15 minute data and just use the hourly data.

From E-Mails (see input_data folder):
* „Value“: Abfluss: Q  [m³/s]; „runoffvalue“: Abflussspende: q  [l / (s . km²)]
* Die Wasserstände sind in absoluten Höhen [cmNHN] angegeben, d.h. die Pegelnullpunkte wurden schon berücksichtigt. Falls Sie trotzdem noch die Pegelnullpunkte benötigen, werde ich Ihnen eine Liste zusenden.
* Code 104 setzt sich zusammen aus: Wertetyp 100 (=kontinuierliche Datenpunkte) + Interpolationstyp 4 (=konstant seit dem vorigen Zeitstempel)

In [43]:
raw_meta_all = pd.read_excel(input_path / "KI-HoPe-LKN-SH.ods")
# strip whitespace from column names
raw_meta_all.columns = raw_meta_all.columns.str.strip()
raw_meta_all["Pegel Nr"] = raw_meta_all["Pegel Nr"].astype(str)

# also strip whitespace from the values in columns "Name" and "Gewaesser"
raw_meta_all["Name"] = raw_meta_all["Name"].str.strip()
raw_meta_all["Gewaesser"] = raw_meta_all["Gewaesser"].str.strip()

raw_meta_all.head()

,Pegel Nr,Name,Gewaesser,x (utm),y (utm),Bemerkung,EZG [km²]
0,114117,Bad Bramstedt/Osterau,Osterau,32559109,5974766,NaN,172.0
1,114116,Bad Bramstedt/Schmalfelder Au,Schmalfelder Au,32558466,5973796,NaN,180.0
2,114120,Brachenfeld,Schwale,32566390,5992817,NaN,73.0
3,114121,Brokstedt,Brokstedter Au,32553170,5983256,NaN,96.0
4,114079,Bünningstedt,Hunnau,32580274,5950513,NaN,64.0


In [25]:
ids_q = sorted([p.name.split("_")[0] for p in (input_path / "Q_und_W" / "Q" / "Cont.60").glob("*.csv")])
ids_w = sorted([p.name.split("_")[0] for p in (input_path / "Q_und_W" / "W" / "Cont.60.Abs").glob("*.csv")])

if ids_q == ids_w:
    ids = ids_q
    print(f"Total number of IDs: {len(ids)}")
    # check if all IDs are in the meta data
    if sorted(raw_meta_all[" Pegel Nr "].astype(str)) == ids:
        print("All IDs are in the meta data.")
else:
    raise ValueError("IDs of Q and W do not match.")


Total number of IDs: 33
All IDs are in the meta data.


In [53]:
def parse_station_data(id: str) -> pd.DataFrame:
    # get filenames
    if id == "114064":
        file_q = input_path / "Q_und_W" / "Q" / "Cont.60" / f"{id}_60m.Cmd.RunOff.(H2).csv"
    elif id == "114069":
        file_q = input_path / "Q_und_W" / "Q" / "Cont.60" / f"{id}_60m.Cmd.(H2).csv"
    else:
        file_q = input_path / "Q_und_W" / "Q" / "Cont.60" / f"{id}_60m.Cmd.RunOff.csv"
        
    file_w = input_path / "Q_und_W" / "W" / "Cont.60.Abs" / f"{id}_60m.Cmd.Abs.csv"

    # read q data for the station
    data_q = pd.read_csv(file_q, sep=";", parse_dates=["tstamp"])
    data_q = data_q.rename(columns={"tstamp": "date", "value": "discharge_vol_obs"})
    data_q = data_q[["date", "discharge_vol_obs"]] # keep only time and discharge columns

    # read w data for the station
    data_w = pd.read_csv(file_w, sep=";", parse_dates=["tstamp"])
    data_w = data_w.rename(columns={"tstamp": "date", "value": "water_level_obs"})
    data_w = data_w[["date", "water_level_obs"]] # keep only time and water level columns

    # merge q and w data
    data = pd.merge(data_q, data_w, on="date", how="outer")

    #! Important: for now we set water level to NaN, as water level values are absolute values [cmNHN], see above, we need gauge elevation information first, update when received
    data["water_level_obs"] = None

    # transform date time to UTC+0 (data is in UTC+1)
    data["date"] = data["date"] - pd.Timedelta(hours=1)
    data["date"] = data["date"].dt.tz_localize("UTC")

    return data

parse_station_data(ids[3])

,date,discharge_vol_obs,water_level_obs
0,1984-10-31 23:00:00+00:00,NaN,None
1,1984-11-01 00:00:00+00:00,0.437,None
2,1984-11-01 01:00:00+00:00,0.437,None
3,1984-11-01 02:00:00+00:00,0.437,None
4,1984-11-01 03:00:00+00:00,0.437,None
...,...,...,...
343316,2023-12-31 19:00:00+00:00,2.181,None
343317,2023-12-31 20:00:00+00:00,2.122,None
343318,2023-12-31 21:00:00+00:00,2.122,None
343319,2023-12-31 22:00:00+00:00,2.122,None


## Check metadata

In [41]:
def check_metadata_consistency(camelsp_meta, raw_meta, id: str) -> dict:
    """
    Check consistency of metadata fields between raw and camelsp sources.
    """
    inconsistencies = {}
    
    fields = {
        "provider_id": {
            "camelsp": "provider_id",
            "raw": "Pegel Nr"
        },
        "gauge_name": {
            "camelsp": "gauge_name",
            "raw": "Name"
        },
        "waterbody": {
            "camelsp": "waterbody_name",
            "raw": "Gewaesser"
        },
        "area": {
            "camelsp": "area",
            "raw": "EZG [km²]"
        },
    }

    for field_key, field_info in fields.items():
        raw_val = raw_meta[field_info["raw"]].values[0]
        camelsp_val = camelsp_meta[field_info["camelsp"]].values[0]
        
        if raw_val != camelsp_val:
            if id in inconsistencies:
                inconsistencies[id].update({field_key: {
                    "raw": raw_val,
                    "camelsp": camelsp_val
                }})
            else:
                inconsistencies[id] = {field_key: {
                    "raw": raw_val,
                    "camelsp": camelsp_val
                }}
    
    return inconsistencies

In [44]:
camelsp_meta_all = get_metadata()

metadata_inconsistencies = {}

not_in_camelsp = []

for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    try:
        gauge_id = get_nuts_id_from_provider_id(id, "DEA", add_missing=False)
    except ValueError:
        not_in_camelsp.append(id)
        continue

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]
    raw_meta = raw_meta_all[raw_meta_all["Pegel Nr"] == id]

    # check the consistency of the metadata
    try:
        inconsistencies = check_metadata_consistency(camelsp_meta, raw_meta, id)
    except IndexError:
        print(f"Error for station {id}")
        continue
    if inconsistencies:
        metadata_inconsistencies.update(inconsistencies)

metadata_inconsistencies

# make it a panads DataFrame with the station_id as index
pd.DataFrame(metadata_inconsistencies).T

  0%|          | 0/33 [00:00<?, ?it/s]

100%|██████████| 33/33 [00:00<00:00, 379.45it/s]


,waterbody,area,gauge_name
114021,"{'raw': 'Osterau', 'camelsp': 'Ostenau'}","{'raw': 71.0, 'camelsp': 71.62}",NaN
114031,NaN,"{'raw': 135.0, 'camelsp': 129.168}",NaN
114042,"{'raw': 'Koseler Au', 'camelsp': 'KoselerAu'}","{'raw': 49.0, 'camelsp': 48.6}",NaN
114064,"{'raw': 'Soholmer Au', 'camelsp': 'SoholmerAu'}","{'raw': 352.0, 'camelsp': 342.308}",NaN
114065,NaN,NaN,"{'raw': 'Sorgbrueck', 'camelsp': 'Sorgbrück'}"
114068,"{'raw': 'Todenbütteler Au', 'camelsp': 'Todenb...",NaN,NaN
114075,"{'raw': 'Loiter Au', 'camelsp': 'LoiterAu'}",NaN,NaN
114079,NaN,"{'raw': 64.0, 'camelsp': 61.49}",NaN
114091,NaN,"{'raw': 137.0, 'camelsp': 136.61}","{'raw': 'Quellenthal/Beste', 'camelsp': 'Quell..."
114094,NaN,"{'raw': 335.0, 'camelsp': 337.39}",NaN


Areas are slightly off, but when comparing to the Pegel information online, the old camelsp metadata seems to be correct / the same as they communicate on their platform (hsi-sh.de/pegel/).  
Other stuff are mostly Umlaute (they are only in camelsp metadata, which is good) and missing whitespaces (whitespaces are missing in camelsp metadata).  
All in all, I think we can just use the old camelsp / CAMELS-DE metadata, information seems to be more correct then the new metadata, just the name formatting is nicer in the new metadata, but not that important.

In [48]:
# check that all stations are also in camelsp
for id in ids:
    if id not in camelsp_meta_all["provider_id"].values:
        print(f"Station {id} is not in camelsp.")

# make the above a one liner
if len([id for id in ids if id not in camelsp_meta_all["provider_id"].values]) > 0:
    print(f"Some stations are not in camelsp: {[id for id in ids if id not in camelsp_meta_all['provider_id'].values]}")
else:
    print("All stations are in camelsp.")

All stations are in camelsp.


## Run for all stations

In [54]:
for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    gauge_id = get_nuts_id_from_provider_id(id, "DEF", add_missing=True)

    # parse data for the station
    data = parse_station_data(id)

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]

    # set metadata values from camelsp metadata
    station_meta = {
        "gauge_id": camelsp_meta["camels_id"].values[0],
        "provider_id": camelsp_meta["provider_id"].values[0],
        "gauge_name": camelsp_meta["gauge_name"].values[0],
        "waterbody_name": camelsp_meta["waterbody_name"].values[0],
        "federal_state": camelsp_meta["federal_state"].values[0],
        "gauge_lon": camelsp_meta["lon"].values[0],
        "gauge_lat": camelsp_meta["lat"].values[0],
        "gauge_easting": camelsp_meta["x"].values[0],
        "gauge_northing": camelsp_meta["y"].values[0],
        "gauge_elev_metadata": camelsp_meta["gauge_elevation"].values[0],
        "area_metadata": camelsp_meta["area"].values[0],
        "part_of_camelsp": True,
    }
    
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(raw_meta)
    

100%|██████████| 33/33 [00:50<00:00,  1.52s/it]
